# 经验贝叶斯：泊松-伽马模型的收缩估计

> 对应代码：`MathStatsCode/code_in_notes/Chap.17/empirical_bayes_poisson_gamma.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**实验目的**：对层次模型 $x_i|\lambda_i\sim\mathcal{P}(\lambda_i)$、$\lambda_i\sim\Gamma(\alpha,\beta)$，用极大似然估计超参数 $\alpha,\beta$（即边缘似然函数），再以后验均值预测每个个体的 $\lambda_i$，观察经验贝叶斯估计的收缩效应。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.17`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致。


In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.17")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


**产生与例中相同的模拟数据**

生成 100 个个体：$\lambda_i$ 来自 $\Gamma(5,2)$（形状 5、尺度 2，均值 $5\times2=10$），$x_i$ 来自以 $\lambda_i$ 为强度的泊松分布。固定种子以保证结果可复现。


In [2]:
%%stata
clear all
set obs 100
set seed 15634145
gen lambda = rgamma(5,2)
gen x = rpoisson(lambda)



. clear all

. set obs 100
Number of observations (_N) was 0, now 100.

. set seed 15634145

. gen lambda = rgamma(5,2)

. gen x = rpoisson(lambda)

. 


**定义计算对数似然函数的程序**

边缘似然函数（把不可观测的 $\lambda_i$ 积分掉之后）的对数为
$$L(\alpha,\beta|x)=\sum_{i=1}^N\Big[\ln\Gamma(x_i+\alpha)-\ln\Gamma(\alpha)+x_i\ln\beta-(x_i+\alpha)\ln(1+\beta)\Big]+C.$$
`ml model d0` 是最一般的（无导数）极大似然估计方式：我们只需写一个计算对数似然的程序 `eb_pg`。程序中：
- `args todo b lnf` 是固定写法，`b` 为参数向量，`lnf` 为要返回的对数似然；
- `mleval` 取出两个方程中的参数；由于 $\alpha,\beta>0$，我们实际估计的是 $\ln\alpha$ 与 $\ln\beta$（方程名 `lalpha`、`lbeta`），程序内用 `exp()` 变回原参数，从而避免优化过程中参数落入负数区域；
- `mlsum` 把每个观测的对数似然累加为总对数似然。

注意 `program define ... end` 必须放在同一个 `%%stata` 单元中运行。


In [3]:
%%stata
cap drop program eb_pg
program define eb_pg
	args todo b lnf
	tempvar la lb
	mleval `la' = `b', eq(1)
	mleval `lb' = `b', eq(2)
	mlsum `lnf' = lngamma(x+exp(`la')) - lngamma(exp(`la')) + x*(`lb') - (x+exp(`la'))*log(1+exp(`lb'))
end



. cap drop program eb_pg

. program define eb_pg
  1.         args todo b lnf
  2.         tempvar la lb
  3.         mleval `la' = `b', eq(1)
  4.         mleval `lb' = `b', eq(2)
  5.         mlsum `lnf' = lngamma(x+exp(`la')) - lngamma(exp(`la')) + x*(`lb')
>  - (x+exp(`la'))*log(1+exp(`lb'))
  6. end

. 


**极大似然估计**

`ml model d0 eb_pg (lalpha: x = ) (lbeta: x = )` 设定模型：两个方程都只有常数项（即待估参数 $\ln\alpha$ 与 $\ln\beta$），`x` 是数据变量；`ml maximize` 执行数值最大化。


In [4]:
%%stata
ml model d0 eb_pg (lalpha: x = ) (lbeta: x = )
ml maximize



. ml model d0 eb_pg (lalpha: x = ) (lbeta: x = )



. ml maximize



Initial:      Log likelihood =  987.95891
Alternative:  Log likelihood =   1284.606
Rescale:      Log likelihood =  1455.3359
Rescale eq:   Log likelihood =  1455.3359


Iteration 0:  Log likelihood =  1455.3359  
Iteration 1:  Log likelihood =  1471.7479  
Iteration 2:  Log likelihood =  1472.3307  
Iteration 3:  Log likelihood =  1472.3324  
Iteration 4:  Log likelihood =  1472.3324  



                                                           Number of obs = 100
                                                           Wald chi2(0)  =   .
Log likelihood = 1472.3324                                 Prob > chi2   =   .



------------------------------------------------------------------------------
             | Coefficient  Std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
lalpha       |
       _cons |   1.475513   .2035686     7.25   0.000     1.076525      1.8745
-------------+----------------------------------------------------------------
lbeta        |
       _cons |   .8749099   .2113766     4.14   0.000     .4606194      1.2892
------------------------------------------------------------------------------

. 


**计算a=αβ以及b=αβ^2**

我们关心的是先验分布的均值 $a=\alpha\beta$ 与方差 $b=\alpha\beta^2$（即 $\lambda_i$ 在总体中的均值与方差）。`nlcom` 用 delta 方法计算这两个非线性组合及其标准误。


In [5]:
%%stata
nlcom (a: exp([lalpha]_cons + [lbeta]_cons)) (b: exp([lalpha]_cons + 2*[lbeta]_cons))



           a: exp([lalpha]_cons + [lbeta]_cons)
           b: exp([lalpha]_cons + 2*[lbeta]_cons)

------------------------------------------------------------------------------
             | Coefficient  Std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
           a |      10.49   .5970924    17.57   0.000      9.31972    11.66028
           b |   25.16194    5.86871     4.29   0.000     13.65948     36.6644
------------------------------------------------------------------------------


**预测每个个体的lambda**

经验贝叶斯预测使用后验均值 $\widehat{\Ep(\lambda_i|\alpha,\beta,x)}=\frac{\hat{\beta}(x_i+\hat{\alpha})}{1+\hat{\beta}}=x_i+\frac{\widehat{\alpha\beta}-x_i}{1+\hat{\beta}}$：在实际观测 $x_i$ 的基础上向总体均值收缩。


In [6]:
%%stata
gen lambda_hat = (x+exp([lalpha]_cons))*exp([lbeta]_cons)/(1+exp([lbeta]_cons))


**对OHIE数据做同样的分析**

载入 OHIE 数据中每个人实验前一年去急诊室的次数 `num_visit_pre_cens_ed`，删除缺失值后改名为 `x`，用同一个程序 `eb_pg` 进行估计。


In [7]:
%%stata
clear
import delimited "../datasets/OHIE_ED_visit.csv"
keep if !missing(num_visit_pre_cens_ed)
rename num_visit_pre_cens_ed x



. clear

. import delimited "../datasets/OHIE_ED_visit.csv"


(encoding automatically selected: ISO-8859-1)


(5 vars, 24,646 obs)

. keep if !missing(num_visit_pre_cens_ed)
(12 observations deleted)

. rename num_visit_pre_cens_ed x



. 


In [8]:
%%stata
ml model d0 eb_pg (lalpha: x = ) (lbeta: x = )
ml maximize
nlcom (a: exp([lalpha]_cons + [lbeta]_cons)) (b: exp([lalpha]_cons + 2*[lbeta]_cons))



. ml model d0 eb_pg (lalpha: x = ) (lbeta: x = )

. ml maximize



Initial:      Log likelihood = -15292.596
Alternative:  Log likelihood = -16026.895
Rescale:      Log likelihood = -14546.208


Rescale eq:   Log likelihood = -13691.686


Iteration 0:  Log likelihood = -13691.686  


Iteration 1:  Log likelihood = -12897.092  
Iteration 2:  Log likelihood = -12848.293  


Iteration 3:  Log likelihood = -12848.291  


Iteration 4:  Log likelihood = -12848.291  

                                                        Number of obs = 24,634
                                                        Wald chi2(0)  =      .
Log likelihood = -12848.291                             Prob > chi2   =      .

------------------------------------------------------------------------------
             | Coefficient  Std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
lalpha       |
       _cons |  -1.286324   .0185111   -69.49   0.000    -1.322605   -1.250043
-------------+----------------------------------------------------------------
lbeta        |
       _cons |   1.030313   .0232815    44.25   0.000     .9846824    1.075944
------------------------------------------------------------------------------

. nlcom (a: exp([lalpha]_cons + [lbeta]_cons)) (b: exp([lalpha]_cons + 2*[lbeta
> ]_cons))

           a: exp([lalpha]_cons + [lbeta]


------------------------------------------------------------------------------
             | Coefficient  Std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
           a |   .7741333   .0109306    70.82   0.000     .7527098    .7955569
           b |   2.169078   .0732407    29.62   0.000     2.025529    2.312627
------------------------------------------------------------------------------

. 


**预测每个个体的lambda并列出结果**

同样计算后验均值预测。`bysort x: gen first=(_n==1)` 标记每个取值的第一条观测，`list ... if first & x<=17` 列出 $x=0,1,\dots,17$ 对应的预测值——可以看到预测值相比实际观测值明显向均值收缩。


In [9]:
%%stata
gen lambda_hat = (x+exp([lalpha]_cons))*exp([lbeta]_cons)/(1+exp([lbeta]_cons))
bysort x: gen first = (_n==1)
list x lambda_hat if first & x<=17, noobs



. gen lambda_hat = (x+exp([lalpha]_cons))*exp([lbeta]_cons)/(1+exp([lbeta]_cons
> ))

. bysort x: gen first = (_n==1)

. list x lambda_hat if first & x<=17, noobs

  +---------------+
  |  x   lambda~t |
  |---------------|
  |  0   .2036152 |
  |  1   .9405918 |
  |  2   1.677568 |
  |  3   2.414545 |
  |  4   3.151522 |
  |---------------|
  |  5   3.888498 |
  |  6   4.625475 |
  |  7   5.362452 |
  |  8   6.099428 |
  |  9   6.836405 |
  |---------------|
  | 10   7.573381 |
  | 11   8.310358 |
  | 12   9.047335 |
  | 13   9.784311 |
  | 14   10.52129 |
  |---------------|
  | 15   11.25826 |
  | 16   11.99524 |
  | 17   12.73222 |
  +---------------+

. 
